In [ ]:
from astropy.io import fits
import sys
import pandas as pd
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy import units as u 

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
from dataloc import *
from bgs_helpers import *
import catalog_definitions as cat
from groupcatalog import *
from plotting import LogLgal_labels, LOG_LGAL_MAX_TIGHT

%load_ext autoreload
%autoreload 2

%load_ext autoreload
%autoreload 2
#####################################################################
# Trying to build a group catalog from BGS DR3 Mocks
# This is to help the DR3 GQP Key Projects on galaxy environments
#####################################################################

In [ ]:
# Cubic box. First step
cubic_path = '/global/cfs/cdirs/desi/users/vtorresg/astra-mocks-kp/data/cubicbox/BGS_box_ph000.fits'
with fits.open(cubic_path) as hdul:
    print(hdul[1].columns)
    print(len(hdul[1].data))

In [ ]:
# This is the is the cutsky mock and Y5 footprint
path = '/global/cfs/cdirs/desi/users/vtorresg/astra-mocks-kp/data/cutsky/z/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000.fits'
#path = '/global/cfs/cdirs/desi/users/vtorresg/astra-mocks-kp/data/cutsky/z_cosmo/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000.fits' # No RSD

# just print columns names
with fits.open(path) as hdul:
    print(hdul[1].columns)
    print(len(hdul[1].data))



In [ ]:
# This file should be the one before fiber assignment happens, but after cutsky, Y5 footprint and imaging mask. Not sure where 4 million galaxies went from the previous step...
fa_info_path = '/global/cfs/cdirs/desi/users/vtorresg/astra-mocks-kp/data/LSScats/FA_files/forFA0.fits'

with fits.open(fa_info_path) as hdul:
    print(hdul[1].columns)
    print(len(hdul[1].data))

In [ ]:
# Final one:  Mock with cutsky to DR3 footprint, RSD, and fiber assignment
# But I need to add back the lost galaxies and give the fake photo-z... I can just add gaussian noise around the true redshift
final_path = '/global/cfs/cdirs/desi/users/vtorresg/astra-mocks-kp/data/LSScats/z/BGS_ANY_clustering.dat.fits'

with fits.open(final_path) as hdul:
    print(hdul[1].columns)
    print(len(hdul[1].data))

## Roadmap: from these mocks to group catalogs

1. **Establish provenance and selection.** Verify that the box, both cut skies, pre-fiber targets and clustering catalog share a realization and compatible IDs. Determine what `IN_Y5`, imaging masks, target bits, tiling and clustering cuts actually mean. A Y5 flag is not automatically a DR3 footprint. The tests below quantify overlaps, not necessarily the cause of each loss.
2. **Choose the analysis sample and its parent.** Match the intended data release, area, magnitude/redshift cuts, mask and tile/pass requirements. A clustering catalog is a selected spectroscopic sample, not automatically a complete parent or a pure fiber-assignment flag. Find a full/assignment-level mock catalog or upstream selection history if needed to distinguish unassigned targets, redshift failures and other cuts. Keep the larger neighbor sample where the data pipeline requires it.
3. **Carry truth through unique joins.** Check `TARGETID` uniqueness and agreement of positions and photometry before joining. Confirm the scope of the halo key (`FILE_NUM`, `BOX_INDEX`, `HALO_INDEX`) and the meaning/units of `HALO_MASS` and `CEN` with the mock producer; repeated halo indices and periodic copies must not be conflated. Preserve cosmological and RSD redshifts and truth separately from observables.
4. **Model incomplete observations.** On the appropriate parent and footprint, attach observed redshifts and classify missing objects using assignment/quality information, not the complement of the clustering catalog alone. Test an all-truth baseline, observed-only baseline, and the same nearest-neighbor / photo-z-plus treatment used for data. A seeded Gaussian photo-z is only a toy: calibrate bias, scatter, tails, outliers, redshift/magnitude/color dependence and failure rate to data, and avoid using truth in inference except for validation.
5. **Adapt preprocessing rather than assuming FITS compatibility.** The draft `BGSMockGroupCatalog` now processes the clustering file as an observed-only sample; see the README for its required area and magnitude limit and its approximations. `BGSGroupCatalog.pre_process_BGS` expects a richer merged data schema, including tile coverage, `PROB_OBS`, derived properties and photo-z. Extending the mock to restore missing galaxies will require deciding which equivalents can be measured or modeled. Recompute redshift-dependent properties after assigning redshifts; verify magnitude/k-correction conventions and cosmology.
6. **Configure and run the finder.** Produce its ordered input (RA, DEC, redshift, log luminosity, and for flux-limited mode VMAX, with optional color/other fields as configured), plus an aligned ID/truth sidecar. Compute VMAX and effective footprint consistently, choose an appropriate halo mass function and parameter/calibration set, and document units and cosmology. The README warns the volume-limited path has not recently been exercised; do not assume a periodic-box catalog can be fed directly to the sky-coordinate finder.
7. **Validate and deliver.** Run small reproducible tests first, then compare recovered groups to truth by membership: completeness/purity, central/satellite classification, halo mass bias/scatter and richness, versus redshift, magnitude, mass, footprint and missing-redshift status. Check clustering and selection effects against data and across realizations; record seeds, provenance, cuts, configuration, joins and output schema. Account for edge effects and the distinction between truth and inferred halo definitions.

## Mock inspection (no catalog construction)

These diagnostics use full `TARGETID` arrays and sorted joins; the files contain tens of millions of rows, so allow several GB of RAM and substantial I/O. They read the original FITS files without writing to them. Sampling is only used for matched-property comparisons and plots; overlap counts use the full catalogs. Rerun the cells in order. A failure on duplicate IDs is intentional: don't interpret a many-to-many match as a valid catalog join.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

mock_root = Path('/global/cfs/cdirs/desi/users/vtorresg/astra-mocks-kp/data')
mock_files = {
    'box': mock_root / 'cubicbox/BGS_box_ph000.fits',
    'cut_rsd': mock_root / 'cutsky/z/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000.fits',
    'cut_cosmo': mock_root / 'cutsky/z_cosmo/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000.fits',
    'pre_fa': mock_root / 'LSScats/FA_files/forFA0.fits',
    'clustering': mock_root / 'LSScats/z/BGS_ANY_clustering.dat.fits',
}
missing_files = [str(p) for p in mock_files.values() if not p.is_file()]
if missing_files:
    raise FileNotFoundError('Missing mock files:\n' + '\n'.join(missing_files))

# Headers and schema only: box coordinates/units may differ from cut-sky fields.
for label, filename in mock_files.items():
    with fits.open(filename, memmap=True) as hdus:
        hdu = hdus[1]
        print(f'\n{label}: {filename}\n  rows={hdu.header["NAXIS2"]:,}, extension={hdu.name}')
        print('  columns:', ', '.join(hdu.columns.names))
        units = {c.name: c.unit for c in hdu.columns if c.unit}
        print('  declared units:', units or 'none')
        print('  selected header metadata:', {k: hdu.header[k] for k in ('COSMO', 'H0', 'OMEGAM', 'BOXSIZE') if k in hdu.header})

In [ ]:
# Keep the memory-mapped HDUs open while accessing their columns.
# On rerun, close any handles from the previous invocation first.
if 'mock_hdus' in globals():
    for hdus in mock_hdus.values():
        hdus.close()
mock_hdus = {name: fits.open(mock_files[name], memmap=True)
             for name in ('cut_rsd', 'pre_fa', 'clustering')}
cut = mock_hdus['cut_rsd'][1].data
pre = mock_hdus['pre_fa'][1].data
clustering = mock_hdus['clustering'][1].data

def sorted_unique_ids(table, label):
    ids = np.asarray(table['TARGETID'])
    order = np.argsort(ids)
    sorted_ids = ids[order]
    duplicates = np.count_nonzero(sorted_ids[1:] == sorted_ids[:-1])
    print(f'{label}: {len(ids):,} rows; {duplicates:,} duplicate ID occurrences')
    if duplicates:
        raise ValueError(f'{label} TARGETID is not unique; investigate before joining')
    return sorted_ids, order

def lookup_rows(query_ids, sorted_ids, order, chunk_size=1_000_000):
    """Row in the indexed table for each query ID, or -1 if absent."""
    rows = np.full(len(query_ids), -1, dtype=np.int64)
    for start in range(0, len(query_ids), chunk_size):
        query = query_ids[start:start + chunk_size]
        pos = np.searchsorted(sorted_ids, query)
        valid = pos < len(sorted_ids)
        valid[valid] = sorted_ids[pos[valid]] == query[valid]
        dest = rows[start:start + len(query)]
        dest[valid] = order[pos[valid]]
    return rows

cut_ids, cut_order = sorted_unique_ids(cut, 'RSD cut sky')
pre_ids, pre_order = sorted_unique_ids(pre, 'pre-FA')
cl_ids, cl_order = sorted_unique_ids(clustering, 'clustering')
del cl_ids, cl_order  # only uniqueness is needed for this table

# Parent row for each child; -1 is an unmatched child, not an array index.
pre_to_cut = lookup_rows(pre['TARGETID'], cut_ids, cut_order)
cl_to_pre = lookup_rows(clustering['TARGETID'], pre_ids, pre_order)
cut_has_pre = np.zeros(len(cut), dtype=bool)
cut_has_pre[pre_to_cut[pre_to_cut >= 0]] = True
pre_has_cl = np.zeros(len(pre), dtype=bool)
pre_has_cl[cl_to_pre[cl_to_pre >= 0]] = True
print(f'pre-FA matched to cut sky: {np.count_nonzero(pre_to_cut >= 0):,} / {len(pre):,}')
print(f'cut sky absent from pre-FA: {np.count_nonzero(~cut_has_pre):,} / {len(cut):,}')
print(f'clustering matched to pre-FA: {np.count_nonzero(cl_to_pre >= 0):,} / {len(clustering):,}')
print(f'pre-FA absent from clustering: {np.count_nonzero(~pre_has_cl):,} / {len(pre):,}')
print('Unmatched children may indicate different IDs or provenance; investigate before constructing a parent.')

In [ ]:
# Full-catalog counts show whether Y5 alone explains the first loss.
def counts_by_flag(values, present, label):
    print(f'\n{label}: value, total, present in next file, absent')
    for value in np.unique(values):
        mask = values == value
        total = np.count_nonzero(mask)
        kept = np.count_nonzero(mask & present)
        print(f'  {value}: {total:,}, {kept:,}, {total - kept:,}')

counts_by_flag(cut['IN_Y5'], cut_has_pre, 'cut sky IN_Y5 -> pre-FA')
counts_by_flag(pre['IN_Y5'], pre_has_cl, 'pre-FA IN_Y5 -> clustering')
# Masks/target bits are descriptive here: their definitions and pipeline cuts must be verified.
for column in ('MASKBITS', 'NOBS_G', 'NOBS_R', 'NOBS_Z', 'BGS_TARGET'):
    counts_by_flag(pre[column], pre_has_cl, f'pre-FA {column} -> clustering')

# Compare properties only on an unbiased, reproducible sample of matched IDs.
rng = np.random.default_rng(20261110)
sample_size = 100_000
def sample_matched(parent_rows, size=sample_size):
    matched = np.flatnonzero(parent_rows >= 0)
    return rng.choice(matched, size=min(size, len(matched)), replace=False)

def differences(label, left, right):
    delta = np.asarray(left, dtype=float) - np.asarray(right, dtype=float)
    finite = delta[np.isfinite(delta)]
    if finite.size == 0:
        print(f'{label}: no finite differences')
    else:
        print(f'{label}: finite={len(finite):,}/{len(delta):,}; '
              f'min, median, 99% |delta|, max = '
              f'{np.min(finite):.6g}, {np.median(finite):.6g}, '
              f'{np.quantile(np.abs(finite), .99):.6g}, {np.max(finite):.6g}')

def compare_pair(child, parent, parent_rows, label):
    ci = sample_matched(parent_rows)
    pi = parent_rows[ci]
    print(f'\n{label}: {len(ci):,} sampled matched IDs')
    # Wrap RA differences at 0/360 degrees.
    dra = (np.asarray(child['RA'][ci]) - np.asarray(parent['RA'][pi]) + 180) % 360 - 180
    differences('RA difference [deg]', dra, np.zeros(len(ci)))
    for col in ('DEC', 'R_MAG_APP', 'R_MAG_ABS', 'G_R_OBS'):
        differences(f'{col} child - parent', child[col][ci], parent[col][pi])
    return ci, pi

pre_i, cut_i = compare_pair(pre, cut, pre_to_cut, 'pre-FA vs cut sky')
differences('pre RSDZ - cut Z', pre['RSDZ'][pre_i], cut['Z'][cut_i])
differences('pre TRUEZ - cut Z', pre['TRUEZ'][pre_i], cut['Z'][cut_i])
cl_i, pre_i_cl = compare_pair(clustering, pre, cl_to_pre, 'clustering vs pre-FA')
differences('clustering Z - pre RSDZ', clustering['Z'][cl_i], pre['RSDZ'][pre_i_cl])
differences('clustering Z - pre TRUEZ', clustering['Z'][cl_i], pre['TRUEZ'][pre_i_cl])
print('Small residuals support a correspondence, but do not prove how a catalog was selected.')

In [ ]:
# Independently test whether the alternative cut sky supplies cosmological redshifts.
with fits.open(mock_files['cut_cosmo'], memmap=True) as cosmo_hdus:
    cosmo = cosmo_hdus[1].data
    cosmo_ids, cosmo_order = sorted_unique_ids(cosmo, 'cosmological cut sky')
    cosmo_to_cut = lookup_rows(cosmo['TARGETID'], cut_ids, cut_order)
    print(f'cosmological cut sky matched to RSD cut sky: {np.count_nonzero(cosmo_to_cut >= 0):,} / {len(cosmo):,}')
    print(f'RSD cut sky absent from cosmological cut sky: {len(cut) - np.count_nonzero(cosmo_to_cut >= 0):,}')
    # Match the same sampled pre-FA rows to this file; never assume row order.
    cosmo_rows = lookup_rows(pre['TARGETID'][pre_i], cosmo_ids, cosmo_order)
    good = cosmo_rows >= 0
    print(f'sampled pre-FA rows matched to cosmological cut sky: {good.sum():,} / {len(pre_i):,}')
    differences('cosmo Z_COSMO - pre TRUEZ', cosmo['Z_COSMO'][cosmo_rows[good]], pre['TRUEZ'][pre_i[good]])
    differences('cosmo Z_COSMO - pre RSDZ', cosmo['Z_COSMO'][cosmo_rows[good]], pre['RSDZ'][pre_i[good]])
    differences('cosmo DEC - pre DEC [deg]', cosmo['DEC'][cosmo_rows[good]], pre['DEC'][pre_i[good]])
    del cosmo_ids, cosmo_order, cosmo_to_cut, cosmo_rows, cosmo

# Reproducible, independently sampled parent distributions: absence from clustering
# is NOT necessarily a fiber-assignment failure. Histograms are normalized by sample size.
def sample_mask(mask):
    indices = np.flatnonzero(mask)
    return rng.choice(indices, size=min(sample_size, len(indices)), replace=False)

kept = sample_mask(pre_has_cl)
missing = sample_mask(~pre_has_cl)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, label in ((kept, 'in clustering'), (missing, 'absent from clustering')):
    for ax, col, bins in zip(axes, ('RSDZ', 'R_MAG_APP', 'G_R_OBS'),
                             (np.linspace(0, 0.6, 61), np.linspace(14, 22, 65), np.linspace(-0.5, 2, 51))):
        values = np.asarray(pre[col][idx], dtype=float)
        values = values[np.isfinite(values)]
        if values.size:
            ax.hist(values, bins=bins, weights=np.full(values.size, 1 / values.size),
                    histtype='step', label=f'{label} (sample n={len(values):,})')
        ax.set_xlabel(col)
        ax.set_ylabel('fraction per bin (fixed display range)')
        ax.legend()
plt.tight_layout()
plt.show()

# Example sky distribution, not an area estimate or an authoritative footprint mask.
fig, ax = plt.subplots(figsize=(10, 4))
for idx, label in ((kept, 'in clustering'), (missing, 'absent from clustering')):
    ax.scatter(pre['RA'][idx], pre['DEC'][idx], s=1, alpha=0.15, label=label, rasterized=True)
ax.set(xlabel='RA [deg]', ylabel='DEC [deg]', title='Sampled pre-FA sky positions')
ax.legend(markerscale=5)
plt.show()
print('When finished, close the memory-mapped FITS files with the next cell.')

In [ ]:
for hdus in mock_hdus.values():
    hdus.close()

# Processing for final stage mock

In [ ]:
groupcat = BGSMockGroupCatalog(mock_files['clustering'], 19.5, 1.0)
groupcat.preprocess()

In [ ]:
groupcat.run_group_finder()

In [ ]:
groupcat.postprocess()

In [ ]:
serialize(groupcat)

In [ ]:
groupcat = deserialize(BGSMockGroupCatalog(mock_files['clustering'], 19.5, 1.0))

In [ ]:
plt.figure()
plt.plot(LogLgal_labels, groupcat.fsatr,  color='r', label='Quiescent', markersize=6)
plt.plot(LogLgal_labels, groupcat.fsatb,  color='b', label='Star-forming', markersize=6)
plt.xlabel('log$(L_{\mathrm{gal}}~/~[L_{\odot}~h^{-2}])$')
plt.ylabel('$f_{\mathrm{sat}}$')
plt.legend()
plt.xlim(8,LOG_LGAL_MAX_TIGHT)
plt.ylim(0.0, 1.0)
plt.twiny()
plt.xticks(np.arange(-23, -10, 1))
plt.xlim(log_solar_L_to_abs_mag_r(8), log_solar_L_to_abs_mag_r(LOG_LGAL_MAX_TIGHT))
plt.xlabel("$M_r$ - 5log(h)")
plt.tight_layout()

In [ ]:
df = groupcat.all_data
df['ABS_MAG_R'] = log_solar_L_to_abs_mag_r(df['LOGLGAL'])

# Compare ABS_MAG_R which is app mag -> abs mag we did to the original R_MAG_ABS_MOCK from mock
delta = df['ABS_MAG_R'] -  df['R_MAG_ABS_MOCK']
plt.hist(delta, bins=40)